# 08 - Change Over Time Analysis

This notebook analyzes how key business metrics change over time.

Focus areas:
- Monthly sales trends
- Year-over-year growth
- Moving average sales
- Cumulative revenue
- Customer activity over time
- Seasonal sales patterns

In [0]:
%sql
/*
Monthly Sales Trend
-------------------
Purpose:
    Analyze sales, customers, orders, and quantity over time at month level.
*/

SELECT
    DATE_TRUNC('MONTH', order_date) AS order_month,
    SUM(sales_amount) AS total_sales,
    COUNT(DISTINCT order_number) AS total_orders,
    COUNT(DISTINCT customer_key) AS total_customers,
    SUM(quantity) AS total_quantity
FROM datawarehouseanalytics_gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', order_date)
ORDER BY order_month;

In [0]:
%sql
/*
Month-over-Month Sales Change
Purpose:
    Compare each month's sales with the previous month to measure growth
    or decline over time.
*/

WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('MONTH', order_date) AS order_month,
        SUM(sales_amount) AS total_sales
    FROM datawarehouseanalytics_gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY DATE_TRUNC('MONTH', order_date)
)

SELECT
    order_month,
    total_sales,
    LAG(total_sales) OVER (ORDER BY order_month) AS previous_month_sales,
    total_sales - LAG(total_sales) OVER (ORDER BY order_month) AS sales_change,
    ROUND(
        (total_sales - LAG(total_sales) OVER (ORDER BY order_month)) * 100.0
        / LAG(total_sales) OVER (ORDER BY order_month),
        2
    ) AS sales_change_percentage
FROM monthly_sales
ORDER BY order_month;

In [0]:
%sql
/*
Year-over-Year Sales Comparison
Purpose:
    Compare annual sales with the previous year to measure business growth.
*/

WITH yearly_sales AS (
    SELECT
        YEAR(order_date) AS order_year,
        SUM(sales_amount) AS total_sales,
        COUNT(DISTINCT order_number) AS total_orders,
        SUM(quantity) AS total_quantity
    FROM datawarehouseanalytics_gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY YEAR(order_date)
)

SELECT
    order_year,
    total_sales,
    LAG(total_sales) OVER (ORDER BY order_year) AS previous_year_sales,
    total_sales - LAG(total_sales) OVER (ORDER BY order_year) AS yearly_sales_change,
    ROUND(
        (total_sales - LAG(total_sales) OVER (ORDER BY order_year)) * 100.0
        / LAG(total_sales) OVER (ORDER BY order_year),
        2
    ) AS yearly_growth_percentage,
    total_orders,
    total_quantity
FROM yearly_sales
ORDER BY order_year;

In [0]:
%sql
/*
Cumulative Sales Over Time

Purpose:
    Calculate running total sales to understand how revenue accumulates
    across the historical period.
*/

WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('MONTH', order_date) AS order_month,
        SUM(sales_amount) AS total_sales
    FROM datawarehouseanalytics_gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY DATE_TRUNC('MONTH', order_date)
)

SELECT
    order_month,
    total_sales,
    SUM(total_sales) OVER (
        ORDER BY order_month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sales
FROM monthly_sales
ORDER BY order_month;

In [0]:
%sql
/*
Three-Month Moving Average

Purpose:
    Smooth monthly sales fluctuations by calculating a rolling
    three-month average.
*/

WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('MONTH', order_date) AS order_month,
        SUM(sales_amount) AS total_sales
    FROM datawarehouseanalytics_gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY DATE_TRUNC('MONTH', order_date)
)

SELECT
    order_month,
    total_sales,
    ROUND(
        AVG(total_sales) OVER (
            ORDER BY order_month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS three_month_moving_avg
FROM monthly_sales
ORDER BY order_month;

In [0]:
%sql
/*
Seasonal Sales Pattern

Purpose:
    Aggregate sales by month of year to identify seasonal patterns
    across the full dataset.
*/

SELECT
    MONTH(order_date) AS month_number,
    DATE_FORMAT(order_date, 'MMM') AS month_name,
    SUM(sales_amount) AS total_sales,
    COUNT(DISTINCT order_number) AS total_orders,
    SUM(quantity) AS total_quantity
FROM datawarehouseanalytics_gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY 
    MONTH(order_date),
    DATE_FORMAT(order_date, 'MMM')
ORDER BY month_number;